In [37]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import torch

from sklearn.impute import SimpleImputer

In [ ]:
# SEED = 123
 
# random.seed(SEED)
# np.random.seed(SEED)
# tf.random.set_seed(SEED)
# torch.manual_seed(SEED)
 
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(SEED)
 
# # =====================================================
# # Load PBC competing risks dataset
# # =====================================================
 
# data = pd.read_csv("pbc2.csv")
 
# data = data.drop(
#     columns=["id", "sno.", "year", "status2"],
#     errors="ignore"
# )
 
# Event indicator
# 0 = censored, 1 = death, 2 = transplant
# event_type = np.where(
#     data["status"] == "dead",
#     1,
#     np.where(data["status"] == "transplanted", 2, 0)
# )
 
# # Continuous variables
# cont_cols = [
#     "age", "serBilir", "serChol", "albumin", "alkaline",
#     "SGOT", "platelets", "prothrombin", "histologic"
# ]
 
# x_cont = data[cont_cols].replace("NA", np.nan).astype(float)
 
# mean_imputer = SimpleImputer(strategy="mean")
# x_cont_imputed = mean_imputer.fit_transform(x_cont)
 
# # Categorical variables
# cat_cols = ["sex", "drug", "ascites", "hepatomegaly", "spiders", "edema"]
 
# x_cat = data[cat_cols].fillna("missing")
 
# cat_imputer = SimpleImputer(strategy="most_frequent")
# x_cat_imputed = cat_imputer.fit_transform(x_cat)
 
# x_cat_df = pd.DataFrame(x_cat_imputed, columns=cat_cols)
 
# x_cat_encoded = pd.get_dummies(
#     x_cat_df,
#     drop_first=True
# )
 
# x_cat_encoded = x_cat_encoded.loc[
#     :,
#     ~x_cat_encoded.columns.str.contains("_missing")
# ]
 
# # Time in days
# T_obs = data["years"].astype(float).values * 365.25
 
# valid = ~np.isnan(T_obs)

In [56]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

# =====================================================
# 1. Load PBC competing risks dataset
# =====================================================
data = pd.read_csv("pbc2.csv")

# Drop unnecessary tracking and duplicate status columns
data = data.drop(columns=["id", "sno.", "year", "status2"], errors="ignore")

# Event indicator: 0 = censored, 1 = death, 2 = transplant
event_type = np.where(
    data["status"] == "dead", 1,
    np.where(data["status"] == "transplanted", 2, 0)
)

# Time in days
t_obs = data["years"].astype(float).values * 365.25

# =====================================================
# 2. Process Continuous Variables
# =====================================================
cont_cols = [
    "age", "serBilir", "serChol", "albumin", "alkaline",
    "SGOT", "platelets", "prothrombin", "histologic"
]

x_cont = data[cont_cols].replace("NA", np.nan).astype(float)
mean_imputer = SimpleImputer(strategy="mean")
x_cont_imputed = pd.DataFrame(
    mean_imputer.fit_transform(x_cont), 
    columns=cont_cols
)

# =====================================================
# 3. Process Categorical Variables
# =====================================================
cat_cols = ["sex", "drug", "ascites", "hepatomegaly", "spiders", "edema"]

x_cat = data[cat_cols].fillna("missing")
cat_imputer = SimpleImputer(strategy="most_frequent")
x_cat_imputed = cat_imputer.fit_transform(x_cat)
x_cat_df = pd.DataFrame(x_cat_imputed, columns=cat_cols)

# One-hot encode and drop the 'missing' indicator columns
x_cat_encoded = pd.get_dummies(x_cat_df, drop_first=True)
x_cat_encoded = x_cat_encoded.loc[:, ~x_cat_encoded.columns.str.contains("_missing")]

# Convert booleans to integers (0/1) for neural network compatibility
x_cat_encoded = x_cat_encoded.astype(int)

# =====================================================
# 4. Combine and Export
# =====================================================
# Concatenate continuous features, encoded categorical features, time, and event
final_df = pd.concat([x_cont_imputed, x_cat_encoded], axis=1)
final_df["time"] = t_obs
final_df["event"] = event_type

# Filter out any rows where observation time is NaN
valid_idx = ~np.isnan(final_df["time"])
final_df = final_df[valid_idx]

# Export to CSV
export_path = "pbc_competing_risks_processed.csv"
final_df.to_csv(export_path, index=False)
print(f"Data successfully processed and saved to '{export_path}'")

Data successfully processed and saved to 'pbc_competing_risks_processed.csv'


In [58]:
final_df.head()

,age,serBilir,serChol,albumin,alkaline,SGOT,platelets,prothrombin,histologic,sex_male,drug_placebo,ascites_Yes,hepatomegaly_Yes,spiders_Yes,edema_edema despite diuretics,edema_edema no diuretics,time,event
0,58.766838,14.5,261.00000,2.60,1718.0,138.0,190.0,12.2,4.0,0,0,1,1,1,1,0,400.010952,1
1,58.766838,21.3,320.47153,2.94,1612.0,6.2,183.0,11.2,4.0,0,0,1,1,1,1,0,400.010952,1
2,56.447815,1.1,302.00000,4.14,7395.0,113.5,221.0,10.6,3.0,0,0,0,1,1,0,0,5169.141523,0
3,56.447815,0.8,320.47153,3.60,2107.0,139.5,188.0,11.0,3.0,0,0,0,1,1,0,0,5169.141523,0
4,56.447815,1.0,320.47153,3.55,1711.0,144.2,161.0,11.6,3.0,0,0,0,1,1,0,0,5169.141523,0


In [59]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1945 entries, 0 to 1944
Data columns (total 18 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   age                            1945 non-null   float64
 1   serBilir                       1945 non-null   float64
 2   serChol                        1945 non-null   float64
 3   albumin                        1945 non-null   float64
 4   alkaline                       1945 non-null   float64
 5   SGOT                           1945 non-null   float64
 6   platelets                      1945 non-null   float64
 7   prothrombin                    1945 non-null   float64
 8   histologic                     1945 non-null   float64
 9   sex_male                       1945 non-null   int32  
 10  drug_placebo                   1945 non-null   int32  
 11  ascites_Yes                    1945 non-null   int32  
 12  hepatomegaly_Yes               1945 non-null   i

In [39]:
df_PBC = pd.read_csv('pbc2.csv')
df_PBC.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1945 entries, 0 to 1944
Data columns (total 21 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sno.          1945 non-null   int64  
 1   id            1945 non-null   int64  
 2   years         1945 non-null   float64
 3   status        1945 non-null   object 
 4   drug          1945 non-null   object 
 5   age           1945 non-null   float64
 6   sex           1945 non-null   object 
 7   year          1945 non-null   float64
 8   ascites       1885 non-null   object 
 9   hepatomegaly  1884 non-null   object 
 10  spiders       1887 non-null   object 
 11  edema         1945 non-null   object 
 12  serBilir      1945 non-null   float64
 13  serChol       1124 non-null   float64
 14  albumin       1945 non-null   float64
 15  alkaline      1885 non-null   float64
 16  SGOT          1945 non-null   float64
 17  platelets     1872 non-null   float64
 18  prothrombin   1945 non-null 

In [40]:
df_PBC = pd.read_csv("pbc2.csv")

df_PBC["event"] = np.where(
    df_PBC["status"] == "dead",
    1,
    np.where(
        df_PBC["status"] == "transplanted",
        2,
        0
    )
)

df_PBC[["status", "event"]].head()

,status,event
0,dead,1
1,dead,1
2,alive,0
3,alive,0
4,alive,0


In [41]:
df_PBC.head(10)

,sno.,id,years,status,drug,age,sex,year,ascites,hepatomegaly,...,serBilir,serChol,albumin,alkaline,SGOT,platelets,prothrombin,histologic,status2,event
0,1,1,1.095170,dead,D-penicil,58.766838,female,0.000000,Yes,Yes,...,14.5,261.0,2.60,1718.0,138.0,190.0,12.2,4,1,1
1,2,1,1.095170,dead,D-penicil,58.766838,female,0.525682,Yes,Yes,...,21.3,NaN,2.94,1612.0,6.2,183.0,11.2,4,1,1
2,3,2,14.152338,alive,D-penicil,56.447815,female,0.000000,No,Yes,...,1.1,302.0,4.14,7395.0,113.5,221.0,10.6,3,0,0
3,4,2,14.152338,alive,D-penicil,56.447815,female,0.498302,No,Yes,...,0.8,NaN,3.60,2107.0,139.5,188.0,11.0,3,0,0
4,5,2,14.152338,alive,D-penicil,56.447815,female,0.999343,No,Yes,...,1.0,NaN,3.55,1711.0,144.2,161.0,11.6,3,0,0
5,6,2,14.152338,alive,D-penicil,56.447815,female,2.102727,No,Yes,...,1.9,NaN,3.92,1365.0,144.2,122.0,10.6,3,0,0
6,7,2,14.152338,alive,D-penicil,56.447815,female,4.900887,Yes,Yes,...,2.6,230.0,3.32,1110.0,131.8,135.0,11.3,3,0,0
7,8,2,14.152338,alive,D-penicil,56.447815,female,5.889278,Yes,Yes,...,3.6,NaN,2.92,996.0,131.8,100.0,11.5,3,0,0
8,9,2,14.152338,alive,D-penicil,56.447815,female,6.885883,Yes,Yes,...,4.2,NaN,2.73,860.0,145.7,103.0,11.5,3,0,0
9,10,2,14.152338,alive,D-penicil,56.447815,female,7.890702,Yes,Yes,...,3.6,244.0,2.80,779.0,119.0,113.0,11.5,3,0,0


In [45]:
df_PBC = df_PBC.drop(columns=["status", "sno.", "status2", "id"])

In [46]:
df_PBC.head()

,years,drug,age,sex,year,ascites,hepatomegaly,spiders,edema,serBilir,serChol,albumin,alkaline,SGOT,platelets,prothrombin,histologic,event
0,1.095170,D-penicil,58.766838,female,0.000000,Yes,Yes,Yes,edema despite diuretics,14.5,261.0,2.60,1718.0,138.0,190.0,12.2,4,1
1,1.095170,D-penicil,58.766838,female,0.525682,Yes,Yes,Yes,edema despite diuretics,21.3,NaN,2.94,1612.0,6.2,183.0,11.2,4,1
2,14.152338,D-penicil,56.447815,female,0.000000,No,Yes,Yes,No edema,1.1,302.0,4.14,7395.0,113.5,221.0,10.6,3,0
3,14.152338,D-penicil,56.447815,female,0.498302,No,Yes,Yes,No edema,0.8,NaN,3.60,2107.0,139.5,188.0,11.0,3,0
4,14.152338,D-penicil,56.447815,female,0.999343,No,Yes,Yes,No edema,1.0,NaN,3.55,1711.0,144.2,161.0,11.6,3,0


In [48]:
df_PBC.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1945 entries, 0 to 1944
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   years         1945 non-null   float64
 1   drug          1945 non-null   object 
 2   age           1945 non-null   float64
 3   sex           1945 non-null   object 
 4   year          1945 non-null   float64
 5   ascites       1885 non-null   object 
 6   hepatomegaly  1884 non-null   object 
 7   spiders       1887 non-null   object 
 8   edema         1945 non-null   object 
 9   serBilir      1945 non-null   float64
 10  serChol       1124 non-null   float64
 11  albumin       1945 non-null   float64
 12  alkaline      1885 non-null   float64
 13  SGOT          1945 non-null   float64
 14  platelets     1872 non-null   float64
 15  prothrombin   1945 non-null   float64
 16  histologic    1945 non-null   int64  
 17  event         1945 non-null   int32  
dtypes: float64(10), int32(1), in

In [49]:
# Check the unique values in all categorical columns
categorical_cols = df_PBC.select_dtypes(include="object").columns

for col in categorical_cols:
    print(f"\n{col}:")
    print(df_PBC[col].value_counts(dropna=False))


drug:
drug
D-penicil    978
placebo      967
Name: count, dtype: int64

sex:
sex
female    1708
male       237
Name: count, dtype: int64

ascites:
ascites
No     1716
Yes     169
NaN      60
Name: count, dtype: int64

hepatomegaly:
hepatomegaly
No     952
Yes    932
NaN     61
Name: count, dtype: int64

spiders:
spiders
No     1311
Yes     576
NaN      58
Name: count, dtype: int64

edema:
edema
No edema                   1401
edema no diuretics          379
edema despite diuretics     165
Name: count, dtype: int64


In [50]:
import numpy as np
import pandas as pd

# ============================================
# 1. Encode categorical variables
# ============================================

# Drug
df_PBC["drug"] = df_PBC["drug"].map({
    "placebo": 0,
    "D-penicil": 1
})

# Sex
df_PBC["sex"] = df_PBC["sex"].map({
    "female": 0,
    "male": 1
})

# Ascites
df_PBC["ascites"] = df_PBC["ascites"].map({
    "No": 0,
    "Yes": 1
})

# Hepatomegaly
df_PBC["hepatomegaly"] = df_PBC["hepatomegaly"].map({
    "No": 0,
    "Yes": 1
})

# Spiders
df_PBC["spiders"] = df_PBC["spiders"].map({
    "No": 0,
    "Yes": 1
})

# Edema
df_PBC["edema"] = df_PBC["edema"].map({
    "No edema": 0,
    "edema no diuretics": 1,
    "edema despite diuretics": 2
})


# ============================================
# 2. Fill missing numerical values
# ============================================

numerical_cols = [
    "years",
    "age",
    "year",
    "serBilir",
    "serChol",
    "albumin",
    "alkaline",
    "SGOT",
    "platelets",
    "prothrombin"
]

for col in numerical_cols:
    df_PBC[col] = df_PBC[col].fillna(df_PBC[col].median())


# ============================================
# 3. Fill missing categorical values
# ============================================

categorical_cols = [
    "drug",
    "sex",
    "ascites",
    "hepatomegaly",
    "spiders",
    "edema"
]

for col in categorical_cols:
    df_PBC[col] = df_PBC[col].fillna(df_PBC[col].mode()[0])


# ============================================
# 4. Make sure there are NO missing values
# ============================================

print("Missing values per column:")
print(df_PBC.isna().sum())

print("\nTotal missing values:")
print(df_PBC.isna().sum().sum())


# ============================================
# 5. Check the final dataset
# ============================================

print("\nDataset information:")
df_PBC.info()

Missing values per column:
years           0
drug            0
age             0
sex             0
year            0
ascites         0
hepatomegaly    0
spiders         0
edema           0
serBilir        0
serChol         0
albumin         0
alkaline        0
SGOT            0
platelets       0
prothrombin     0
histologic      0
event           0
dtype: int64

Total missing values:
0

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1945 entries, 0 to 1944
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   years         1945 non-null   float64
 1   drug          1945 non-null   int64  
 2   age           1945 non-null   float64
 3   sex           1945 non-null   int64  
 4   year          1945 non-null   float64
 5   ascites       1945 non-null   float64
 6   hepatomegaly  1945 non-null   float64
 7   spiders       1945 non-null   float64
 8   edema         1945 non-null   int64  
 9   serB

In [52]:
df_PBC.rename(columns={'years': 'time'}, inplace=True)

In [53]:
df_PBC.head()

,time,drug,age,sex,year,ascites,hepatomegaly,spiders,edema,serBilir,serChol,albumin,alkaline,SGOT,platelets,prothrombin,histologic,event
0,1.095170,1,58.766838,0,0.000000,1.0,1.0,1.0,2,14.5,261.0,2.60,1718.0,138.0,190.0,12.2,4,1
1,1.095170,1,58.766838,0,0.525682,1.0,1.0,1.0,2,21.3,281.0,2.94,1612.0,6.2,183.0,11.2,4,1
2,14.152338,1,56.447815,0,0.000000,0.0,1.0,1.0,0,1.1,302.0,4.14,7395.0,113.5,221.0,10.6,3,0
3,14.152338,1,56.447815,0,0.498302,0.0,1.0,1.0,0,0.8,281.0,3.60,2107.0,139.5,188.0,11.0,3,0
4,14.152338,1,56.447815,0,0.999343,0.0,1.0,1.0,0,1.0,281.0,3.55,1711.0,144.2,161.0,11.6,3,0


In [55]:
df_PBC.to_csv("pbc_competing_risks.csv")